In [27]:
# parser two with file handling
from scapy.all import PcapReader, UDP
import pynmea2
import json
import os

def load_pcap(filepath, max_sentences=1000):
    """Extract raw NMEA sentences from a pcap file, streaming packet by packet."""
    sentences = []
    with PcapReader(filepath) as reader:
        for packet in reader:
            if len(sentences) >= max_sentences:
                break
            if UDP in packet:
                try:
                    payload = bytes(packet[UDP].payload).decode("ascii").strip().rstrip("\x00")
                    if payload.startswith("$"):
                        sentences.append(payload)
                except UnicodeDecodeError:
                    pass
    return sentences

def parse_sentence(raw):
    try:
        return pynmea2.parse(raw)
    except pynmea2.ParseError as e:
        print(f"[WARN] Could not parse: {raw!r} — {e}\n")
        return None

def assess_quality(rmc):
    if rmc.status == "V":
        return 0.0
    score = 1.0
    hdop = getattr(rmc, "horizontal_dil", None)
    if hdop:
        hdop = float(hdop)
        if hdop > 2.0:
            score -= 0.3
        elif hdop > 1.0:
            score -= 0.1
    return round(score, 2)

def build_track(rmc, hdt=None):
    confidence = assess_quality(rmc)
    return {
        "trackId":       f"TRK-{rmc.timestamp}",
        "sentenceType":  "RMC+HDT" if hdt else "RMC",
        "type":          "SURFACE",
        "source":        "NMEA_GNSS",
        "timestamp":     rmc.datetime.isoformat() if rmc.datetime else None,
        "position": {
            "lat": rmc.latitude  if rmc.lat != None else None,
            "lon": rmc.longitude if rmc.lon != None else None,
        },
        "velocity": {
            "speedKnots":       float(rmc.spd_over_grnd) if rmc.spd_over_grnd != None else None,
            "courseOverGround": float(rmc.true_course)   if rmc.true_course != None else None,
            "trueHeading":      float(hdt.heading)       if hdt and hdt.heading != None else None,
        },
        "pntConfidence": confidence,
        "gnssStatus":    rmc.status,
    }



In [28]:
def debug(sentences):
# Debug: count sentence types
    from collections import Counter
    type_counts = Counter()
    for raw in sentences:
        try:
            sentence_id = raw[1:6]  # e.g. GPRMC, GPHDT, GPGSV
            type_counts[sentence_id] += 1
        except:
            pass
    print("Sentence types found:")
    for k, v in type_counts.most_common():
        print(f"  {k}: {v}")
    print()

In [29]:
def process_pcap(filepath, output_file="tracks_output.json", max_sentences=1000):
    sentences = load_pcap(filepath, max_sentences=max_sentences)
    print(f"✓ Extracted {len(sentences)} NMEA sentences (capped at {max_sentences})\n")
    
    tracks = []
    last_hdt = None
    for raw in sentences:
                # Only process HDT and RMC sentences, skip everything else
        sentence_id = raw[3:6]  # extracts 'HDT' or 'RMC' from '$GPHDT' or '$GPRMC'
        if sentence_id not in ("HDT", "RMC"):
            continue
            
        msg = parse_sentence(raw)
        if msg is None:
            continue
        if isinstance(msg, pynmea2.HDT):
            last_hdt = msg
        elif isinstance(msg, pynmea2.RMC):
            track = build_track(msg, hdt=last_hdt)
            tracks.append(track)
    
    if os.path.exists(output_file): #if file exists, delete
        os.remove(output_file)
    with open(output_file, "w") as f:
        json.dump(tracks, f, indent=2, default=str)
    print(f"✓ Processed {len(tracks)} track updates")
    print(f"✓ Results written to {output_file}")
    return tracks



In [30]:
# Run code
if __name__ == "__main__":
    results = process_pcap("kraken_data/multicast_AdvNavData-2026-05-08_07-40-14_capture.pcap")
    print(f"\n✓ Processed {len(results)} track updates")

✓ Extracted 1000 NMEA sentences (capped at 1000)

Sentence types found:
  GPRMC: 284
  GPHDT: 283
  GPGGA: 114
  GPVTG: 113
  GPHDM: 113
  GPGSV: 70
  GPZDA: 23

✓ Processed 284 track updates
✓ Results written to tracks_output.json

✓ Processed 284 track updates


In [32]:
"""
NMEA → STANAG Track WebSocket Streamer
Replays a pcap file in real-time, streaming parsed track objects over WebSocket.

Usage:
    python nmea_streamer.py                        # stream from pcap
    python nmea_streamer.py --test                 # test with a client in the same script
"""

import asyncio
import json
import time
import argparse
from datetime import timezone
from scapy.all import PcapReader, UDP
import pynmea2
import websockets

# ─── Config ────────────────────────────────────────────────────────────────────
PCAP_FILE   = "kraken_data/multicast_AdvNavData-2026-05-08_07-40-14_capture.pcap"
HOST        = "localhost"
PORT        = 8765
SPEED_FACTOR = 1.0   # 1.0 = real time, 2.0 = double speed, 0.0 = no delay (fire as fast as possible)
# ───────────────────────────────────────────────────────────────────────────────


# ─── Parser (unchanged from your notebook) ─────────────────────────────────────

def assess_quality(rmc):
    if rmc.status == "V":
        return 0.0
    score = 1.0
    hdop = getattr(rmc, "horizontal_dil", None)
    if hdop:
        hdop = float(hdop)
        if hdop > 2.0:
            score -= 0.3
        elif hdop > 1.0:
            score -= 0.1
    return round(score, 2)


def build_track(rmc, hdt=None):
    confidence = assess_quality(rmc)
    return {
        "trackId":       f"TRK-{rmc.timestamp}",
        "sentenceType":  "RMC+HDT" if hdt else "RMC",
        "type":          "SURFACE",
        "source":        "NMEA_GNSS",
        "timestamp":     rmc.datetime.isoformat() if rmc.datetime else None,
        "position": {
            "lat": rmc.latitude  if rmc.lat  is not None else None,
            "lon": rmc.longitude if rmc.lon  is not None else None,
        },
        "velocity": {
            "speedKnots":       float(rmc.spd_over_grnd) if rmc.spd_over_grnd is not None else None,
            "courseOverGround": float(rmc.true_course)   if rmc.true_course   is not None else None,
            "trueHeading":      float(hdt.heading)       if hdt and hdt.heading is not None else None,
        },
        "pntConfidence": confidence,
        "gnssStatus":    rmc.status,
    }


# ─── Generator: yield (delay_seconds, track_dict) ──────────────────────────────

def pcap_track_generator(filepath):
    """
    Reads a pcap file and yields (wall_delay, track) tuples.
    wall_delay is how long to wait before sending this track (based on pcap timestamps).
    """
    last_hdt       = None
    last_pkt_time  = None

    with PcapReader(filepath) as reader:
        for packet in reader:
            if UDP not in packet:
                continue

            try:
                payload = bytes(packet[UDP].payload).decode("ascii").strip().rstrip("\x00")
            except UnicodeDecodeError:
                continue

            if not payload.startswith("$"):
                continue

            sentence_id = payload[3:6]
            if sentence_id not in ("HDT", "RMC"):
                continue

            try:
                msg = pynmea2.parse(payload)
            except pynmea2.ParseError:
                continue

            # Calculate real-time delay between packets
            pkt_time = float(packet.time)
            if last_pkt_time is None:
                delay = 0.0
            else:
                delay = pkt_time - last_pkt_time
            last_pkt_time = pkt_time

            if isinstance(msg, pynmea2.HDT):
                last_hdt = msg
                continue  # HDT alone doesn't produce a track; wait for RMC

            if isinstance(msg, pynmea2.RMC):
                track = build_track(msg, hdt=last_hdt)
                yield delay, track


# ─── WebSocket Server ───────────────────────────────────────────────────────────

connected_clients = set()


async def broadcast(message: str):
    if connected_clients:
        await asyncio.gather(
            *[client.send(message) for client in connected_clients],
            return_exceptions=True
        )


async def handle_client(websocket):
    connected_clients.add(websocket)
    print(f"[+] Client connected: {websocket.remote_address}  (total: {len(connected_clients)})")
    try:
        await websocket.wait_closed()
    finally:
        connected_clients.discard(websocket)
        print(f"[-] Client disconnected  (total: {len(connected_clients)})")


async def stream_pcap(filepath, speed_factor=1.0):
    """Replay pcap in real-time and broadcast each track to all connected clients."""
    print(f"[*] Starting pcap replay: {filepath}")
    print(f"[*] Speed factor: {speed_factor}x")

    track_count = 0
    for delay, track in pcap_track_generator(filepath):
        # Wait proportional to real packet gap
        if speed_factor > 0 and delay > 0:
            await asyncio.sleep(delay / speed_factor)

        message = json.dumps(track, default=str)
        await broadcast(message)
        track_count += 1

        if track_count % 50 == 0:
            print(f"[~] Streamed {track_count} tracks | clients: {len(connected_clients)}")

    print(f"[✓] Replay complete. Total tracks streamed: {track_count}")

    # After replay ends, send a sentinel so clients know the stream is done
    await broadcast(json.dumps({"event": "END_OF_STREAM", "totalTracks": track_count}))


async def main(filepath, speed_factor=1.0):
    async with websockets.serve(handle_client, HOST, PORT):
        print(f"[✓] WebSocket server listening on ws://{HOST}:{PORT}")
        print(f"[*] Waiting 1s for clients to connect before starting replay...")
        await asyncio.sleep(1)
        await stream_pcap(filepath, speed_factor=speed_factor)
        # Keep server alive after replay in case clients reconnect
        print("[*] Replay done — server still running. Ctrl-C to stop.")
        await asyncio.Future()  # run forever


# ─── Optional: built-in test client ────────────────────────────────────────────

async def test_client():
    """A simple client that connects and prints the first 10 messages."""
    await asyncio.sleep(0.5)  # give server a moment to start
    uri = f"ws://{HOST}:{PORT}"
    print(f"\n[TEST CLIENT] Connecting to {uri}")
    async with websockets.connect(uri) as ws:
        for i in range(10):
            msg = await ws.recv()
            data = json.loads(msg)
            print(f"[TEST CLIENT] Track #{i+1}: {data.get('trackId')} | "
                  f"pos=({data.get('position', {}).get('lat'):.5f}, "
                  f"{data.get('position', {}).get('lon'):.5f}) | "
                  f"ts={data.get('timestamp')}")


async def run_with_test(filepath, speed_factor):
    await asyncio.gather(
        main(filepath, speed_factor),
        test_client()
    )


# ─── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="NMEA pcap → WebSocket streamer")
    parser.add_argument("--pcap",   default=PCAP_FILE,    help="Path to .pcap file")
    parser.add_argument("--speed",  default=SPEED_FACTOR, type=float, help="Replay speed factor (1.0 = real time)")
    parser.add_argument("--test",   action="store_true",  help="Also run a test client")
    args = parser.parse_args()

    if args.test:
        asyncio.run(run_with_test(args.pcap, args.speed))
    else:
        asyncio.run(main(args.pcap, args.speed))

usage: ipykernel_launcher.py [-h] [--pcap PCAP] [--speed SPEED] [--test]
ipykernel_launcher.py: error: unrecognized arguments: -f /home/salleh-miro/.local/share/jupyter/runtime/kernel-319655d9-ccbe-4e75-b5ac-3c872b961cfb.json


SystemExit: 2

/home/salleh-miro/anaconda3/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
